# Longleaf Pipeline Smoke Test

This notebook is a lightweight validation harness for running the `deception2` pipeline on UNC Longleaf.

It tests:
- environment + GPU visibility
- path resolution on Longleaf vs local paths
- tiny sentence dataset build (`--limit`)
- optional tiny localization run (`RUN_LOCALIZATION=1`)

It writes to a separate smoke-test output tree so you do not overwrite production results.

In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path
from datetime import datetime

def run_cmd(cmd: str, check: bool = True):
    print(f"$ {cmd}")
    proc = subprocess.run(
        cmd,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(proc.stdout)
    if check and proc.returncode != 0:
        raise RuntimeError(f"Command failed ({proc.returncode}): {cmd}")
    return proc

def is_repo_root(p: Path) -> bool:
    return (p / "src").exists() and (p / "BS").exists() and (p / "Gridworld").exists()

candidates = []
if os.environ.get("DECEPTION_ROOT"):
    candidates.append(Path(os.environ["DECEPTION_ROOT"]).expanduser())
candidates.extend([
    Path("/work/users/s/m/smerrill/deception").expanduser(),
    Path.cwd(),
    *Path.cwd().parents,
])

PROJECT_ROOT = None
for cand in candidates:
    if is_repo_root(cand):
        PROJECT_ROOT = cand
        break
    if is_repo_root(cand / "deception2"):
        PROJECT_ROOT = cand / "deception2"
        break

if PROJECT_ROOT is None:
    raise RuntimeError("Could not find deception2 repo root. Set DECEPTION_ROOT explicitly.")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Timestamp:", datetime.now().isoformat(timespec="seconds"))

In [ ]:
run_cmd("hostname", check=False)
run_cmd("whoami", check=False)
run_cmd("pwd", check=False)
run_cmd("python --version", check=False)
run_cmd("which python", check=False)
run_cmd("echo CUDA_VISIBLE_DEVICES=$CUDA_VISIBLE_DEVICES", check=False)
run_cmd("nvidia-smi -L", check=False)
run_cmd("python - <<'PY'\nimport torch\nprint('torch:', torch.__version__)\nprint('cuda_available:', torch.cuda.is_available())\nprint('cuda_devices:', torch.cuda.device_count())\nPY", check=False)


In [ ]:
# Config (override via environment variables before launching Jupyter)
GAME = os.environ.get("GAME", "bs").strip().lower()
if GAME not in {"bs", "gridworld"}:
    raise ValueError(f"Invalid GAME={GAME}")

MODEL_NAME = os.environ.get("MODEL_NAME", "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B")
LABEL_FILTER = os.environ.get("LABEL_FILTER", "truthful_only")
SMOKE_LIMIT = int(os.environ.get("SMOKE_LIMIT", "20"))
N_SAMPLES = int(os.environ.get("N_SAMPLES", "4"))
RUN_LOCALIZATION = os.environ.get("RUN_LOCALIZATION", "0") == "1"

ENV_DIR = "BS" if GAME == "bs" else "Gridworld"
model_tag_base = MODEL_NAME.split("/")[-1]
model_tag_raw = MODEL_NAME.replace("/", "_")

mining_root = Path(os.environ.get("MINING_ROOT", str(PROJECT_ROOT / ENV_DIR / "Results" / "DeceptionMining")))
sentence_smoke_root = Path(os.environ.get("SENTENCE_SMOKE_ROOT", str(PROJECT_ROOT / ENV_DIR / "Results" / "SentencePipeline" / "longleaf_smoke")))

cand_a = mining_root / model_tag_base
cand_b = mining_root / model_tag_raw
if cand_a.exists():
    mining_model_dir = cand_a
elif cand_b.exists():
    mining_model_dir = cand_b
else:
    raise FileNotFoundError(f"Could not find model dir in {mining_root}: {cand_a.name} or {cand_b.name}")

smoke_out_dir = sentence_smoke_root / model_tag_base / f"{LABEL_FILTER}_limit{SMOKE_LIMIT}"
examples_path = smoke_out_dir / "examples.jsonl"
sentences_path = smoke_out_dir / "sentences.jsonl"
loc_jsonl = smoke_out_dir / "localization_smoke.jsonl"
loc_out_dir = smoke_out_dir / "localization_smoke"

print("GAME:", GAME)
print("MODEL_NAME:", MODEL_NAME)
print("LABEL_FILTER:", LABEL_FILTER)
print("SMOKE_LIMIT:", SMOKE_LIMIT)
print("RUN_LOCALIZATION:", RUN_LOCALIZATION)
print("mining_model_dir:", mining_model_dir)
print("smoke_out_dir:", smoke_out_dir)

In [ ]:
build_script = PROJECT_ROOT / "src" / "build_sentence_dataset.py"
loc_script = PROJECT_ROOT / "src" / "sentence_localization_batch.py"

assert build_script.exists(), build_script
assert loc_script.exists(), loc_script

print("build_script:", build_script)
print("loc_script:", loc_script)

cmd = (
    f"{sys.executable} {build_script} "
    f"--input_root {mining_model_dir} "
    f"--out_dir {smoke_out_dir} "
    f"--text_field action_reasoning "
    f"--fallback_text_field action_raw_text "
    f"--label_filter {LABEL_FILTER} "
    f"--limit {SMOKE_LIMIT} "
    f"--include_messages"
)
run_cmd(cmd, check=True)


In [ ]:
def count_jsonl(path: Path) -> int:
    if not path.exists():
        return 0
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

def label_counts(path: Path):
    counts = {True: 0, False: 0, None: 0}
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            d = rec.get("deceptive")
            if d is True:
                counts[True] += 1
            elif d is False:
                counts[False] += 1
            else:
                counts[None] += 1
    return counts

print("examples_path:", examples_path)
print("sentences_path:", sentences_path)
print("examples rows:", count_jsonl(examples_path))
print("sentences rows:", count_jsonl(sentences_path))
if examples_path.exists():
    print("example label counts:", label_counts(examples_path))


In [ ]:
if RUN_LOCALIZATION:
    loc_out_dir.mkdir(parents=True, exist_ok=True)
    cmd = (
        f"{sys.executable} {loc_script} "
        f"--game {GAME} "
        f"--examples_path {examples_path} "
        f"--sentences_path {sentences_path} "
        f"--jsonl_path {loc_jsonl} "
        f"--out_dir {loc_out_dir} "
        f"--model_name {MODEL_NAME} "
        f"--label_filter all "
        f"--limit 4 "
        f"--n_samples {N_SAMPLES} "
        f"--method adaptive "
        f"--mode prefix "
        f"--log_every 1"
    )
    run_cmd(cmd, check=True)
    print("localization rows:", count_jsonl(loc_jsonl))
else:
    print("RUN_LOCALIZATION=0, skipping localization smoke run.")
    print("Set environment variable RUN_LOCALIZATION=1 before launching Jupyter to enable.")


## Suggested Longleaf Structure

Use this layout under your Longleaf workspace:

- `deception2/` code
- `deception2/BS/Results/...` outputs
- `deception2/Gridworld/Results/...` outputs
- optional `deception2/.cluster/env.sh` for cluster-specific environment setup

High-value refactor target after smoke tests pass:

- remove absolute `/playpen-ssd/...` paths from shell scripts
- use `PROJECT_ROOT="$(cd "$(dirname "$0")/../.." && pwd)"`
- use env vars: `RESULTS_ROOT`, `CONDA_SH`, `MODEL_NAME`, `GPU_IDS`